# BundledSeries Demo

`BundledSeries` is a virtual series that bundles fields from multiple source `TimeSeries`. 
It subscribes to ALL source series and triggers attached indicators when ANY source updates.

**Use cases:**
- Combining price data with volume data from different sources
- Creating indicators that need OHLCV + orderbook data
- Multi-source feature engineering (e.g., VWAP deviation from close price)

In [1]:
import numpy as np
import pandas as pd

from qubx.core.basics import TimestampedDict
from qubx.core.series import (
    TimeSeries, OHLCV, ColumnarSeries, BundledSeries, 
    IndicatorGeneric, Bar
)
from qubx.ta.indicators import sma

## 1. Basic Usage: Bundling Two TimeSeries

Create a bundle from two simple TimeSeries and see how updates propagate.

In [2]:
# Create two source TimeSeries
ts_price = TimeSeries("price", "1h")
ts_volume = TimeSeries("volume", "1h")

# Create a BundledSeries that combines them
bundle = BundledSeries("price_volume", "1h", {
    "price": ts_price,
    "volume": ts_volume
})

print(f"Field names: {bundle.field_names}")
print(f"Fields: {bundle.fields}")

Field names: ['price', 'volume']
Fields: {'price': Series([], Name: price, dtype: float64), 'volume': Series([], Name: volume, dtype: float64)}


In [3]:
# Add some data to both series
base_time = np.datetime64("2024-01-01T00:00:00", "ns")
hour_ns = 60 * 60 * 10**9

for i in range(5):
    t = base_time.item() + i * hour_ns
    ts_price.update(t, 100.0 + i * 5)   # 100, 105, 110, 115, 120
    ts_volume.update(t, 1000.0 + i * 100)  # 1000, 1100, 1200, 1300, 1400

print(f"Bundle length: {len(bundle)}")
print(f"Latest bundle value: {bundle[0]}")

Bundle length: 5
Latest bundle value: {'price': 120.0, 'volume': 1400.0}


In [4]:
# View all bundled values as DataFrame
bundle.to_series()

,price,volume
timestamp,,
2024-01-01 00:00:00,100.0,1000.0
2024-01-01 01:00:00,105.0,1100.0
2024-01-01 02:00:00,110.0,1200.0
2024-01-01 03:00:00,115.0,1300.0
2024-01-01 04:00:00,120.0,1400.0


## 2. Attaching Indicators to BundledSeries

Create custom indicators that work with the bundled dict values using `IndicatorGeneric`.

In [5]:
class PriceVolumeRatio(IndicatorGeneric):
    """
    Example indicator that computes price / volume.
    Receives a dict with 'price' and 'volume' fields.
    """
    
    def calculate(self, time, values, new_item_started):
        price = values.get("price", np.nan)
        volume = values.get("volume", np.nan)
        
        if np.isnan(price) or np.isnan(volume) or volume == 0:
            return np.nan
        
        return price / volume


# Create fresh series and bundle
ts_price = TimeSeries("price", "1h")
ts_volume = TimeSeries("volume", "1h")
bundle = BundledSeries("pv_bundle", "1h", {"price": ts_price, "volume": ts_volume})

# Attach the indicator
ratio = PriceVolumeRatio("pv_ratio", bundle)

# Add data
for i in range(5):
    t = base_time.item() + i * hour_ns
    ts_price.update(t, 100.0 + i * 10)
    ts_volume.update(t, 1000.0)

print(f"Indicator length: {len(ratio)}")
print(f"Latest ratio: {ratio[0]:.4f}")  # 140 / 1000 = 0.14
print(f"All ratios: {[round(ratio[i], 4) for i in range(len(ratio))]}")

Indicator length: 5
Latest ratio: 0.1400
All ratios: [0.14, 0.13, 0.12, 0.11, 0.1]


## 3. Real-World Example: VWAP Deviation

Combine OHLCV close price with VTWAP data to compute price deviation from VWAP.

In [6]:
# Create OHLCV series (price source)
ohlcv = OHLCV("BTCUSDT", "1h")

# Create ColumnarSeries for VTWAP data (like vendor data)
vtwap = ColumnarSeries("vtwap", "1h", ["twap", "vwap"])

# Bundle close price with vwap
bundle = BundledSeries("close_vwap", "1h", {
    "close": ohlcv.close,
    "vwap": vtwap.vwap
})

print(f"Bundle fields: {bundle.field_names}")

Bundle fields: ['close', 'vwap']


In [7]:
class VwapDeviation(IndicatorGeneric):
    """
    Computes (close - vwap) / close as a percentage deviation.
    Positive = price above VWAP, Negative = price below VWAP.
    """
    
    def calculate(self, time, values, new_item_started):
        close = values.get("close", np.nan)
        vwap = values.get("vwap", np.nan)
        
        if np.isnan(close) or np.isnan(vwap) or close == 0:
            return np.nan
        
        return (close - vwap) / close * 100  # As percentage


# Attach indicator
deviation = VwapDeviation("vwap_dev", bundle)

# Simulate data: price starts at 50000, vwap slightly below
for i in range(10):
    t = base_time.item() + i * hour_ns
    price = 50000.0 + i * 50 + np.random.randn() * 10
    
    # Update OHLCV (simplified - just close price)
    ohlcv.update(t, price)
    
    # Update VTWAP (vwap typically lags behind price moves)
    vtwap_val = price - 20 + np.random.randn() * 5  # VWAP ~20 below close
    vtwap.update(TimestampedDict(time=t, data={
        "twap": vtwap_val + 5,
        "vwap": vtwap_val
    }))

print(f"Deviation indicator length: {len(deviation)}")
print(f"Sample deviations: {[round(deviation[i], 3) for i in range(min(5, len(deviation)))]}")

Deviation indicator length: 10
Sample deviations: [0.047, 0.04, 0.038, 0.037, 0.049]


## 4. Key Behaviors

### 4.1 Updates from ANY source trigger the bundle

When either source updates, the bundle gathers latest values from ALL sources and triggers indicators.

In [9]:
# Demo: updates from either source trigger the bundle

class UpdateCounter(IndicatorGeneric):
    """Counts how many times calculate() is called."""
    
    def __init__(self, name, series):
        self.count = 0
        super().__init__(name, series)
    
    def calculate(self, time, values, new_item_started):
        self.count += 1
        return self.count


ts_a = TimeSeries("a", "1m")
ts_b = TimeSeries("b", "1m")
bundle = BundledSeries("ab", "1m", {"a": ts_a, "b": ts_b})
counter = UpdateCounter("counter", bundle)

t0 = np.datetime64("2024-01-01T00:00:00", "ns").item()
minute_ns = 60 * 10**9

# Update source A
ts_a.update(t0, 1.0)
print(f"After A update: counter.count = {counter.count}")

# Update source B (same timestamp)
ts_b.update(t0, 10.0)
print(f"After B update: counter.count = {counter.count}")

# Both sources at next minute
t1 = t0 + minute_ns
ts_a.update(t1, 2.0)
print(f"After A update (t1): counter.count = {counter.count}")
ts_b.update(t1, 20.0)
print(f"After B update (t1): counter.count = {counter.count}")

After A update: counter.count = 1
After B update: counter.count = 2
After A update (t1): counter.count = 3
After B update (t1): counter.count = 4


### 4.2 Late updates from lagging sources are skipped

If one source advances to a new bar while another sends a late update for the previous bar, 
the late update is silently skipped to prevent data corruption.

In [10]:
# Demo: late updates are skipped

ts_fast = TimeSeries("fast", "1h")
ts_slow = TimeSeries("slow", "1h")
bundle = BundledSeries("fast_slow", "1h", {"fast": ts_fast, "slow": ts_slow})

t0 = np.datetime64("2024-01-01T00:00:00", "ns").item()
hour_ns = 60 * 60 * 10**9

# Both at hour 0
ts_fast.update(t0, 1.0)
ts_slow.update(t0, 100.0)

# Both at hour 1
t1 = t0 + hour_ns
ts_fast.update(t1, 2.0)
ts_slow.update(t1, 200.0)

# Fast advances to hour 2
t2 = t0 + 2 * hour_ns
ts_fast.update(t2, 3.0)
print(f"After fast advances to hour 2: bundle length = {len(bundle)}")
print(f"Latest bundle: {bundle[0]}")

# Slow sends LATE update for hour 1 (should be skipped!)
t1_late = t0 + hour_ns + 30 * 60 * 10**9  # 1:30 - still in hour 1
ts_slow.update(t1_late, 250.0)
print(f"After slow's late update: bundle length = {len(bundle)} (unchanged)")
print(f"Latest bundle: {bundle[0]} (fast=3.0 preserved, slow from earlier lookup)")

# Slow catches up to hour 2
ts_slow.update(t2, 300.0)
print(f"After slow catches up: bundle = {bundle[0]}")

After fast advances to hour 2: bundle length = 3
Latest bundle: {'fast': 3.0, 'slow': 200.0}
After slow's late update: bundle length = 3 (unchanged)
Latest bundle: {'fast': 3.0, 'slow': 200.0} (fast=3.0 preserved, slow from earlier lookup)
After slow catches up: bundle = {'fast': 3.0, 'slow': 300.0}


## 5. Advanced: Multi-Field Indicator with State

Create a more complex indicator that maintains internal state across updates.

In [11]:
class VwapSlope(IndicatorGeneric):
    """
    Computes VWAP slope normalized by close price.
    
    Formula: (vwap[t] - vwap[t-lag]) / close[t]
    
    This requires maintaining history of vwap values.
    """
    
    def __init__(self, name, series, lag: int = 5):
        self.lag = lag
        self._vwap_history = []
        super().__init__(name, series)
    
    def calculate(self, time, values, new_item_started):
        vwap = values.get("vwap", np.nan)
        close = values.get("close", np.nan)
        
        if np.isnan(vwap) or np.isnan(close) or close == 0:
            return np.nan
        
        if new_item_started:
            self._vwap_history.append(vwap)
            # Keep only what we need
            if len(self._vwap_history) > self.lag + 1:
                self._vwap_history.pop(0)
        else:
            # Update last value
            if self._vwap_history:
                self._vwap_history[-1] = vwap
        
        # Need lag+1 values to compute slope
        if len(self._vwap_history) <= self.lag:
            return np.nan
        
        vwap_prev = self._vwap_history[-self.lag - 1]
        slope = (vwap - vwap_prev) / close
        return slope * 10000  # In basis points


# Create data sources
ohlcv = OHLCV("BTCUSDT", "1h")
vtwap = ColumnarSeries("vtwap", "1h", ["twap", "vwap"])

# Bundle with selected fields
bundle = BundledSeries("close_vwap", "1h", {
    "close": ohlcv.close,
    "vwap": vtwap.vwap
})

# Attach slope indicator with lag=3
slope = VwapSlope("vwap_slope_3", bundle, lag=3)

# Generate sample data
np.random.seed(42)
price = 50000.0
for i in range(15):
    t = base_time.item() + i * hour_ns
    price += np.random.randn() * 50
    vwap_val = price - 10 + np.random.randn() * 5
    
    ohlcv.update(t, price)
    vtwap.update(TimestampedDict(time=t, data={"twap": vwap_val + 3, "vwap": vwap_val}))

# Display results
print("VWAP Slope (3-period, basis points):")
slope_df = slope.pd()
print(slope_df.dropna())

VWAP Slope (3-period, basis points):
2024-01-01 04:00:00     7.759706
2024-01-01 05:00:00     6.221930
2024-01-01 06:00:00    -9.572796
2024-01-01 07:00:00   -20.567066
2024-01-01 08:00:00   -24.200619
2024-01-01 09:00:00   -36.023297
2024-01-01 10:00:00    -4.217162
2024-01-01 11:00:00     4.513951
2024-01-01 12:00:00    11.420973
2024-01-01 13:00:00   -15.708572
2024-01-01 14:00:00   -21.884215
Name: vwap_slope_3, dtype: float64


## Summary

`BundledSeries` provides:

1. **Multi-source data fusion** - Combine fields from different TimeSeries, OHLCV, ColumnarSeries
2. **Automatic update propagation** - Triggers when ANY source updates
3. **Late update handling** - Silently skips stale updates from lagging sources
4. **Dict-based values** - Indicators receive `{"field1": val1, "field2": val2, ...}`
5. **Works with IndicatorGeneric** - Standard indicator pattern with `calculate(time, values, new_item_started)`

### Typical Usage Pattern

```python
# 1. Create source series
ohlcv = OHLCV("BTCUSDT", "1m")
vendor_data = ColumnarSeries("vendor", "1m", ["metric1", "metric2"])

# 2. Bundle selected fields
bundle = BundledSeries("my_bundle", "1m", {
    "close": ohlcv.close,
    "metric": vendor_data.metric1
})

# 3. Create indicator
class MyIndicator(IndicatorGeneric):
    def calculate(self, time, values, new_item_started):
        close = values["close"]
        metric = values["metric"]
        return some_calculation(close, metric)

indicator = MyIndicator("my_indicator", bundle)

# 4. Stream data - bundle and indicator update automatically
for bar in data_stream:
    ohlcv.update(bar.time, bar.close)
    vendor_data.update(TimestampedDict(time=bar.time, data={...}))
    # indicator[0] now has latest computed value
```